<a href="https://colab.research.google.com/github/Freibott/clasificador-milito/blob/main/clasificador_milito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificador de textos en esperanto: Segunda Guerra Mundial, Guerra Civil Española y Guerras Napoleónicas

Minería de Textos · CUNEF · 2026/2027

## Fase 1 · Dataset

### Exploración de fuentes

**Objetivo.** Antes de construir el dataset comprobamos (30/10/2026) si Vikipedio (Wikipedia en esperanto) tiene texto suficiente para las tres clases. Para ello consultamos su API de MediaWiki en modo de solo lectura. La bitácora completa está en `docs/bitacora_exploracion_fase1.md` del repositorio.

**Por qué estas clases.** Empezamos con WWI, WWII y Guerra Fría, pero a profundidad 1 compartían 32 artículos (sobre todo en 1943–1949 y en el periodo de entreguerras). Eso mete ruido en las etiquetas y limita la precisión. Las sustituimos por tres guerras del mismo tema y con muy pocos artículos en común. Antes descartamos otras candidatas por falta de datos: Vietnam, Afganistán, Aceh, la guerra civil china y Corea.

**Hallazgos (páginas directas, recuento completo).**

| | Segunda Guerra Mundial | Guerra Civil Española | Guerras Napoleónicas |
|---|---|---|---|
| Categoría | `Kategorio:Dua Mondmilito` | `Kategorio:Hispana Enlanda Milito` | `Kategorio:Napoleonaj militoj` |
| Artículos directos | 132 | 54 | 41 |
| Subcategorías | 27 | 4 | 8 |
| Palabras mín / mediana / máx | 12 / 284 / 12.238 | 58 / 386 / 5.050 | 76 / 279 / 1.780 |
| Párrafos (≥10 palabras) por artículo | 25,1 | 14,9 | 7,3 |
| Párrafos en páginas directas | 3.312 | 802 | 298 |

**Solapes entre páginas directas:** ninguno.

**Valoración.** Es viable. Las clases son del mismo tema, como pide el enunciado. La Guerra Civil Española y la Segunda Guerra Mundial son cercanas (1936–1945, fascismo, intervención alemana e italiana), así que el problema no es trivial. Con solo las páginas directas, las Guerras Napoleónicas no llegan a 300 párrafos, por lo que estudiamos las subcategorías.

**Incidencias resueltas.** Los nombres de categoría distinguen mayúsculas. La API de extractos solo devuelve un texto completo por petición. Además, hubo que gestionar el error 429 (demasiadas peticiones) con reintentos y pausas.

### Estudio de profundidad de subcategorías

**Objetivo.** Medimos (30/10/2026) cuántos párrafos únicos se obtienen al bajar 0, 1 o 2 niveles en el árbol de categorías de cada clase. Se descargaron los 1.306 artículos distintos hasta profundidad 2, así que es un recuento completo. El detalle está en `docs/estudio_profundidad_fase1.md` del repositorio.

**Párrafos únicos por clase** (líneas de 10 palabras o más, sin contar los artículos que comparten varias clases):

| Profundidad | Segunda Guerra Mundial | Guerra Civil Española | Guerras Napoleónicas |
|---|---:|---:|---:|
| 0 (páginas directas) | 3.312 | 802 | 298 |
| 1 | 5.371 | 1.689 | 735 |
| 2 | 11.880 | 2.034 | 746 |

**Solapes.** A profundidad 1 solo hay 2 artículos compartidos, ambos entre la Segunda Guerra Mundial y la Guerra Civil Española: Hugo Sperrle e Hispana ekonomia miraklo. A profundidad 2 son 9. Las Guerras Napoleónicas no comparten ninguno.

**Decisiones.**
1. **Profundidad 1.** Es la mínima con la que las Guerras Napoleónicas superan los 300 párrafos (735). La profundidad 2 apenas les aporta (+11) y duplica la Segunda Guerra Mundial.
2. **Excluir:**
   - "Sekvoj de la Dua Mondmilito" y "Frankismo", porque son de posguerra y por ellas entra el solape "Hispana ekonomia miraklo".
   - "Brit-usona milito de 1812", porque es otra guerra.
   - El artículo "Listo de svedoj kiuj mortis en la Hispana Enlanda Milito", porque es una lista de nombres.

   Las subcategorías dudosas (biografías, lugares, obras) se mantienen y su ruido se mide más adelante.
3. **Descartar los artículos presentes en más de una clase.** Tras las exclusiones solo queda uno (Hugo Sperrle).
4. **Equilibrar las clases** recortando cada una al tamaño de la menor (Guerras Napoleónicas, unos 720 párrafos), con muestreo aleatorio y semilla fija. En la Fase 3 se compara con el dataset sin equilibrar.
5. **Conservar números y fechas** por ahora. En la Fase 3 se entrena con y sin ellos, porque los años (1799–1815, 1936–1939, 1939–1945) pueden delatar la clase casi solos.

### Generación del dataset

El código de generación está en el cuaderno, como pide el enunciado, pero se puede desactivar con `GENERAR_DATASET`. Con `False`, el cuaderno carga el CSV ya generado desde el repositorio. Con `True`, lo vuelve a construir desde la API de Vikipedio, lo que tarda varios minutos por las pausas entre peticiones.

In [1]:
# --- Importaciones ---
import re
import time
import random
from collections import deque

import requests
import pandas as pd

# --- Parámetros de ejecución ---
GENERAR_DATASET = True   # True: construir el dataset desde la API · False: cargar el CSV del repo

# --- Fuente y categorías (decisiones del estudio de profundidad) ---
API_URL = "https://eo.wikipedia.org/w/api.php"
CATEGORIAS = {                                   # clase -> categoría raíz (distingue mayúsculas)
    "Segunda Guerra Mundial": "Kategorio:Dua Mondmilito",
    "Guerra Civil Española": "Kategorio:Hispana Enlanda Milito",
    "Guerras Napoleónicas": "Kategorio:Napoleonaj militoj",
}
PROFUNDIDAD = 1                                  # decisión 1
SUBCATS_EXCLUIDAS = {                            # decisión 2
    "Kategorio:Sekvoj de la Dua Mondmilito",     # posguerra
    "Kategorio:Frankismo",                       # posguerra (dictadura 1939-1975)
    "Kategorio:Brit-usona milito de 1812",       # otra guerra
}
ARTICULOS_EXCLUIDOS = {                          # decisión 2
    "Listo de svedoj kiuj mortis en la Hispana Enlanda Milito",  # lista de nombres
}

# --- Limpieza y equilibrado ---
MIN_PALABRAS = 10                                # párrafo = línea con 10 palabras o más
SEMILLA = 42                                     # decisión 4 (reproducibilidad)

# --- Cortesía con la API (evita el error 429) ---
PAUSA = 1.0                                      # segundos entre peticiones
MAX_REINTENTOS = 4
USER_AGENT = "ClasificadorMilito/0.1 (https://github.com/Freibott/clasificador-milito; práctica CUNEF)"

# --- Rutas ---
RUTA_CSV = "data/dataset.csv"
URL_CSV_REMOTO = "https://raw.githubusercontent.com/Freibott/clasificador-milito/main/data/dataset.csv"

In [2]:
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})  # Wikimedia exige identificarse


def consultar_api(params):
    """Hace una petición GET a la API de MediaWiki y devuelve el JSON como diccionario.

    Centraliza la identificación (User-Agent), las pausas entre peticiones y los
    reintentos con espera creciente ante el límite de peticiones (error 429).
    """
    p = dict(params)                 # copia: no modificamos el diccionario de quien llama
    # Establecemos formato json
    p["format"] = "json"
    p["formatversion"] = 2           # MediaWiki: devuelve listas en vez de diccionarios por ID

    for intento in range(MAX_REINTENTOS):
        respuesta = session.get(API_URL, params=p, timeout=30)

        # 429 -> demasiadas peticiones, 5xx -> error del servidor: se reintenta
        if respuesta.status_code == 429 or respuesta.status_code >= 500:
            time.sleep(PAUSA * 2**intento)   # espera creciente (1, 2, 4, 8 s)
            continue

        respuesta.raise_for_status()         # otros errores HTTP (400, 403, 404): no se reintentan
        datos = respuesta.json()

        # MediaWiki responde 200 incluso con errores de la consulta
        if "error" in datos:
            raise RuntimeError(f"Error en la API de MediaWiki: {datos['error']['info']}")

        time.sleep(PAUSA)                    # pausa para no saturar el servidor
        return datos

    # Solo se llega aquí si se agotan todos los intentos
    raise RuntimeError(f"La API falló tras {MAX_REINTENTOS} intentos, último código: {respuesta.status_code}")

Comprobación: la raíz de la Segunda Guerra Mundial debe devolver 161 miembros directos (134 páginas y 27 subcategorías).

In [3]:
datos = consultar_api({
    "action": "query",
    "list": "categorymembers",
    "cmtitle": CATEGORIAS["Segunda Guerra Mundial"],
    "cmlimit": 500,
})
len(datos["query"]["categorymembers"])  # → 161

29

## Fase 2 · NLP

## Fase 3 · Clasificación